<!--
SPDX-FileCopyrightText: Copyright (c) 2026, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# NeMo Fabric quickstart

**NeMo Fabric is one typed interface in front of many agent frameworks.**

Teams build agents on different harnesses -- Hermes Agent, Codex, Claude, and Deep
Agents. Each has its own way to pick a model, wire up tools and skills, launch
the agent, and collect results. NeMo Fabric puts a single contract in front of all of
them:

- You describe an agent **once** as a typed `FabricConfig` -- model, harness,
  tools, skills, environment, telemetry.
- NeMo Fabric resolves that config, launches it on whichever harness you pick, and
  returns a **normalized result** with the same shape no matter which harness
  actually ran.

**Why reach for it:**

- **Swap harnesses without rewriting your app.** Point the same config at a
  different adapter to compare harnesses -- the whole point of the next notebook.
- **One result, artifact, and telemetry contract** across every harness, so your
  downstream code stays stable when the harness changes.
- **Config is data.** An agent is just a typed object, so you can vary the
  model, tools, or skills programmatically.

This notebook is fully self-contained -- every agent is built inline. It works
on a single harness (Hermes Agent) and is organized around one distinction: the calls
that **run** an agent (`run`, `start_runtime`) versus the optional calls that
only **inspect** it (`plan`, `doctor`). We build a config, run it, then look at
the inspection tools.

<div align="center">
<img src="img/fabric-contract.svg" width="620" alt="Where NeMo Fabric sits between your application and the agent harnesses.">
<br><br>
<em>Your app hands NeMo Fabric a <code>FabricConfig</code>. Two calls run the agent (<code>run</code>, <code>start_runtime</code>); two optional calls only inspect it (<code>plan</code>, <code>doctor</code>). NeMo Fabric resolves an adapter, drives the chosen harness, and returns a normalized <code>RunResult</code>.</em>
</div>

## Prerequisites

- Build the SDK and native extension from the repo root: `just build-all`.
- Make `NVIDIA_API_KEY` available. This repo keeps it in a gitignored
  `.env` at the root, which the setup cell loads for you.

Launch with the project interpreter so `nemo_fabric` is importable:
`just notebooks` (or `.venv/bin/jupyter lab`).

Each live-run cell is guarded, so the notebook runs top to bottom even
before Hermes Agent and the API key are set up -- it just tells you what is
missing instead of erroring.

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk upward from ``start`` until a directory holds ``pyproject.toml``."""
    root = start.resolve()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent
    return root


def load_dotenv(path: Path) -> list[str]:
    """Set os.environ from a KEY=VALUE file (honors a leading ``export``).

    Existing environment values win, so an already-exported shell variable is
    never overwritten. Returns the names that were loaded from the file.
    """
    loaded: list[str] = []
    if not path.exists():
        return loaded
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        line = line.removeprefix("export ").strip()
        key, _, value = line.partition("=")
        key = key.strip()
        if key and key not in os.environ:
            os.environ[key] = value.strip().strip("'\"")
            loaded.append(key)
    return loaded


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

# ADAPTER_PYTHON selects the interpreter that runs the Hermes adapter, and is
# optional. When it is unset we default to this interpreter -- which works when
# Hermes and its adapter are installed here (for example via `uv sync
# --all-groups --all-extras`) -- falling back to a dedicated Hermes venv if one
# is present.
if "ADAPTER_PYTHON" not in os.environ:
    _hermes_venv = REPO_ROOT / ".tmp" / "hermes-venv" / "bin" / "python"
    os.environ["ADAPTER_PYTHON"] = str(_hermes_venv) if _hermes_venv.exists() else sys.executable

# Validate ADAPTER_PYTHON, resolving a relative path against the repo root so the
# adapter launches correctly from any working directory.
_adapter_python = os.environ.get("ADAPTER_PYTHON")
HAS_ADAPTER = False
if _adapter_python:
    _adapter_python_path = Path(_adapter_python)
    if not _adapter_python_path.is_absolute():
        _adapter_python_path = REPO_ROOT / _adapter_python_path
    HAS_ADAPTER = _adapter_python_path.is_file()
    os.environ["ADAPTER_PYTHON"] = str(_adapter_python_path)

NATIVE_OK = importlib.util.find_spec("nemo_fabric._native") is not None
HAS_NVIDIA_KEY = bool(os.environ.get("NVIDIA_API_KEY"))
CAN_RUN_HERMES = NATIVE_OK and HAS_NVIDIA_KEY and HAS_ADAPTER

print("repo root      :", REPO_ROOT)
print("native ext     :", "OK" if NATIVE_OK else "MISSING - run `just build-all`")
print("NVIDIA_API_KEY :", "set" if HAS_NVIDIA_KEY else "unset")
print("adapter python :", os.environ["ADAPTER_PYTHON"])
print("live Hermes run:", "ready" if CAN_RUN_HERMES else "skipped (missing prerequisite above)")


## 1. Describe the agent

Everything NeMo Fabric does starts from a `FabricConfig`: a single typed object that
fully describes the agent. The cell below builds a small **code-review** agent,
with the meaning of each block explained inline:

- `metadata` -- the agent's identity.
- `harness` -- which harness to use (`adapter_id`) plus that harness's own
  `settings`, which NeMo Fabric passes through untouched.
- `models` -- named model aliases; `api_key_env` names the credential.
- `runtime` -- the input/output contract and where artifacts are written.

In [ ]:
from nemo_fabric import (
    Fabric,
    FabricConfig,
    HarnessConfig,
    MetadataConfig,
    ModelConfig,
    RuntimeConfig,
)

WORKSPACE = REPO_ROOT / "examples" / "code_review_agent" / "repos" / "my-service"
ARTIFACTS = REPO_ROOT / "examples" / "notebooks" / "artifacts" / "quickstart"

config = FabricConfig(
    # Who the agent is.
    metadata=MetadataConfig(
        name="quickstart-reviewer",
        description="Reviews Python code for correctness issues.",
    ),
    # Which harness runs it. `adapter_id` selects the harness integration;
    # everything under `settings` is handed straight to that harness.
    harness=HarnessConfig(
        adapter_id="nvidia.fabric.hermes",
        resolution="preinstalled",
        settings={
            "workspace": str(WORKSPACE),
            "hermes_home": str(ARTIFACTS / "hermes-home"),
            "base_url": "https://integrate.api.nvidia.com/v1",
            "system_prompt": "You are a code reviewer. Point out correctness bugs and risks, concisely.",
            "max_iterations": 1,
            "max_tokens": 1024,
            "reasoning_config": {"effort": "none"},
            "enabled_toolsets": [],
        },
    ),
    # Named model aliases. `api_key_env` names the environment variable that
    # holds the credential, so the key itself never lives in the config.
    models={
        "default": ModelConfig(
            provider="nvidia",
            model="nvidia/nemotron-3-nano-30b-a3b",
            temperature=0.0,
            api_key_env="NVIDIA_API_KEY",
        )
    },
    # The logical input/output contract and the artifact root for this agent.
    runtime=RuntimeConfig(
        input_schema="chat",
        output_schema="message",
        artifacts=str(ARTIFACTS),
    ),
)
print("configured:", config.metadata.name)

## 2. Run it -- `run()`

**To run an agent you need exactly two things: a `FabricConfig` and `run()` (or
`start_runtime()` for multiple turns).** The `plan()` and `doctor()` calls
further down are optional -- they inspect and diagnose, and never run anything.

| To run your agent | To inspect it (optional) |
| --- | --- |
| `run()`, `start_runtime()` | `plan()`, `doctor()` |

`run()` performs one full lifecycle in a single call: it starts a runtime, sends
your input, collects the result, and stops the runtime. Reach for it for
single-invocation work. It is async, and Jupyter lets you `await` it directly.

It returns a **`RunResult`**: a normalized record of the invocation with the same
shape regardless of harness. Always check `status` and `error` first, then read
`output.response` (the harness's reply). The correlation IDs (`runtime_id`,
`invocation_id`, `request_id`) tie the run to its logs, artifacts, and telemetry.
Here we ask the reviewer to look at a function with a lurking bug, then read the
result back.

In [ ]:
fabric = Fabric()

REVIEW_INPUT = (
    "Review this Python function for correctness issues, and explain any bug you find:\n\n"
    "def average(values):\n"
    "    return sum(values) / len(values)\n"
)

if CAN_RUN_HERMES:
    try:
        result = await fabric.run(config, input=REVIEW_INPUT)
    except Exception as error:
        result = None
        print(f"Live run failed ({type(error).__name__}: {error}).")
    else:
        response = getattr(result.output, "response", result.output)
        print("status     :", result.status)
        print("error      :", result.error)
        print("runtime_id :", result.runtime_id)
        print("invocation :", result.invocation_id)
        print("artifacts  :", len(result.artifacts.artifacts),
              "file(s) under", result.artifacts.root)
        print("\n--- review ---\n")
        print(response)
else:
    result = None
    print("Skipping the live run - a prerequisite above is missing (native build,")
    print("NVIDIA_API_KEY, or a resolvable ADAPTER_PYTHON). See the Hermes quick start.")

## 3. Keep state across turns -- `start_runtime()`

`run()` is stateless: each call is independent. When a later turn depends on an
earlier one, start a **runtime** and send it ordered invocations -- the harness
keeps its conversation state between calls. This is still the *run* path (it
executes your agent), just stateful. Used as an `async with` context manager, the
runtime shuts down automatically; each runtime handles one invocation at a time.

**When to use it:** interactive or iterative work -- follow-up questions,
step-by-step refinement, anything where turn 2 must remember turn 1. Below, the
first turn reviews the function and the second asks the agent to fix *the issue
it just found* -- only possible because the runtime carried that context forward.

In [ ]:
if CAN_RUN_HERMES:
    try:
        async with await fabric.start_runtime(config) as runtime:
            first = await runtime.invoke(input=REVIEW_INPUT)
            second = await runtime.invoke(
                input="Now rewrite the function to fix the issue you just found."
            )
    except Exception as error:
        print(f"Live run failed ({type(error).__name__}: {error}).")
    else:
        print("--- turn 1: review ---\n")
        print(getattr(first.output, "response", None))
        print("\n--- turn 2: fix (uses turn 1's context) ---\n")
        print(getattr(second.output, "response", None))
else:
    print("Skipping multi-turn - same prerequisites as the run above.")

## Inspect and diagnose (optional)

Everything above was the run path. The next two calls are **optional**: they take
the same `FabricConfig` but never run your agent. Use them to understand what
will happen or to catch problems early -- before a run, or on their own, in any
order, or not at all.

### `plan()` -- preview what will run

`plan()` resolves your config and the adapter it maps to, and reports the runtime
capabilities you'll get (streaming, updates, cancellation). It does not start
anything or call the model, so it is instant and needs no credentials.

**When to use it:** to confirm you selected the harness you meant and that an
optional capability you depend on is supported. The `adapter_id` is what you'd
change to run this same agent elsewhere -- exactly what the next notebook does.

In [ ]:
plan = fabric.plan(config)

print("adapter :", plan.adapter.adapter_id)
print("harness :", plan.adapter.harness)
print("in/out  :", plan.config.runtime.input_schema,
      "/", plan.config.runtime.output_schema)
print("caps    : streaming=%s updates=%s cancellation=%s" % (
    plan.capabilities.streaming,
    plan.capabilities.updates,
    plan.capabilities.cancellation,
))

### `doctor()` -- preflight the environment

Where `plan()` proves the *config* is valid, `doctor()` proves the *machine* is
ready: it runs preflight diagnostics against the resolved adapter -- is the
harness installed, are required settings and environment variables present, are
the requested capabilities supported -- returning a report of individual checks
rather than stopping at the first problem.

**When to use it:** *recommended, not required.* Run it before your first run in a
new environment (a fresh container, a teammate's laptop, CI) to catch a missing
install or credential early. You never need it to run a working setup.

In [ ]:
if NATIVE_OK:
    report = await fabric.doctor(config)
    print("overall:", report.status, "\n")
    for check in report.checks:
        print(f"[{check.status:>4}] {check.name}: {check.message}")
else:
    print("Skipping doctor - native extension not built (`just build-all`).")

## Recap

Two buckets, one config. You **ran** the agent with just a `FabricConfig` and
`run()` (plus `start_runtime()` for turns), then used the optional **inspect**
calls, `plan()` and `doctor()`, which never run anything:

- **Run:** `run()`, `start_runtime()` -- execute the agent.
- **Inspect (optional):** `plan()`, `doctor()` -- understand or debug it, in any
  order or not at all.

And notice it all used a **single** harness. Running one agent on one harness is
not the reason to adopt NeMo Fabric -- you could call Hermes Agent directly for that. The
payoff is that the *same* config runs on Codex, Claude, or Deep Agents, and that
you can vary its skills, MCP servers, and telemetry as data, without touching
your application code.

This quickstart built everything inline. **Continue to the advanced notebook,
[`02_variations.ipynb`](02_variations.ipynb)**, which builds on the maintained
[code-review example](../code_review_agent/README.md) to vary the harness and the
agent's configuration.

For the complete API, see the [Python SDK guide](../../docs/sdk/python.mdx).